# Pipeline Completo de Experimentos: Leis de Escala (Scaling Laws)

Este notebook unifica todo o fluxo de trabalho de engenharia de dados, treinamento em Deep Learning e modelagem matemática de Leis de Escala para a classificação de folhas de tomate (10 classes).

### Estrutura do Pipeline Integrado:
1. **Preparação & Particionamento de Dados**: Balanceamento em 1.000 imagens/classe, separação do teste global (15%), splits em Fase 1 (50%) e Fase 2 (50%) e geração de subsets logarítmicos.
2. **Definição da Rede e Otimizadores**: ResNet-18 escalável por largura (fator $\alpha$) instanciada do zero, pesos inicializados via Xavier e otimização por steps com warmup linear + Cosine Annealing.
3. **Orquestração de Experimentos (Grid Search - 140 Rodadas)**: Execução das 5 seeds variando a capacidade do modelo ($M$) e o volume de dados ($N$) na Fase 1.
4. **Ajuste Matemática de Leis de Escala**: Ajuste da equação de Rosenfeld e RandomForestRegressor.
5. **Validação Ground Truth**: Treinamento consolidado em Fase 1+2 e cálculo da margem de erro da predição.

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (320).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (321).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (322).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (324).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (325).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (326).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (327).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (328).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (329).JPG  
  inflating: dataset/processed/train_fase_1/Tomato___Tomato_mosaic_virus/image (33).JPG  
  inflating: dataset/processed/t

## Passo 1: Configuração do Ambiente e Importação de Pacotes

In [44]:
# Instalação e carregamento de dependências standard
!pip install -q pandas numpy matplotlib scipy scikit-learn torch torchvision pillow

import os
import csv
import math
import random
import time
import warnings
from pathlib import Path
from typing import Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import f1_score, precision_score, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

warnings.filterwarnings("ignore")

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    try:
        import torch_directml
        if torch_directml.is_available():
            return torch_directml.device()
    except ImportError:
        pass
    return torch.device("cpu")

device = get_device()
print(f"✓ Ambiente pronto. Dispositivo ativo: {device}")

✓ Ambiente pronto. Dispositivo ativo: cuda


## Passo 2: Montagem do Google Drive (Opcional)
Caso utilize o dataset original armazenado no Drive, descomente e execute a célula abaixo.

In [45]:
# from google.colab import drive
# drive.mount('/content/drive')

## Passo 3: Engenharia de Dados: Compilação e Separação do Dataset
Esta função implementa o balanceamento e o particionamento em diretórios físicos no ambiente local.

In [50]:
import json
import shutil
def process_dataset(input_dir_str: str, output_dir_str: str, seed: int, max_images_per_class: int, test_pct: float):
    input_path = Path(input_dir_str).resolve()
    output_path = Path(output_dir_str).resolve()

    if not input_path.exists():
        print(f"[!] Caminho de entrada não encontrado: {input_path}")
        print("Crie a pasta ou faça upload do dataset antes de prosseguir.")
        return False

    class_dirs = [d for d in input_path.iterdir() if d.is_dir()]
    if not class_dirs:
        raise ValueError(f"Nenhuma pasta de classe encontrada em {input_path}")

    extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG", "*.JPEG")

    metadata = {
        "config": {
            "seed": seed,
            "max_images_per_class": max_images_per_class,
            "test_pct": test_pct,
            "input_dir": str(input_path),
            "output_dir": str(output_path),
        },
        "statistics": {
            "total_images_processed": 0,
            "classes": {},
            "splits": {
                "global_test": 0,
                "train_fase_1": 0,
                "train_fase_2": 0,
                "fase_1_mais_fase_2": 0,
                "subsets_fase_1": {}
            }
        },
        "file_lineage": {}
    }

    cumulative_percentages = [1, 2, 5, 10, 20, 50, 100]
    for pct in cumulative_percentages:
        metadata["statistics"]["splits"]["subsets_fase_1"][f"pct_{pct}"] = 0

    global_test_dir = output_path / "global_test"
    train_fase_1_dir = output_path / "train_fase_1"
    train_fase_2_dir = output_path / "train_fase_2"
    fase_1_plus_2_dir = output_path / "fase_1_mais_fase_2"
    subsets_dir = output_path / "subsets_fase_1"

    # Explicitly create all base split directories
    global_test_dir.mkdir(parents=True, exist_ok=True)
    train_fase_1_dir.mkdir(parents=True, exist_ok=True)
    train_fase_2_dir.mkdir(parents=True, exist_ok=True)
    fase_1_plus_2_dir.mkdir(parents=True, exist_ok=True)
    subsets_dir.mkdir(parents=True, exist_ok=True)

    # Ensure all percentage subdirectories within subsets_fase_1 exist
    for pct in cumulative_percentages:
        (subsets_dir / f"pct_{pct}").mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)

    for class_dir in class_dirs:
        class_name = class_dir.name
        images = []
        for ext in extensions:
            images.extend(list(class_dir.glob(ext)))

        images = sorted(list(set(images)))
        total_found = len(images)

        if total_found == 0:
            continue

        shuffled_original = list(images)
        rng.shuffle(shuffled_original)
        selected_images = shuffled_original[:max_images_per_class]
        actual_class_count = len(selected_images)

        metadata["statistics"]["classes"][class_name] = {
            "available_count": total_found,
            "selected_count": actual_class_count,
            "splits": {
                "global_test": 0,
                "train_fase_1": 0,
                "train_fase_2": 0,
                "fase_1_mais_fase_2": 0,
                "subsets_fase_1": {}
            }
        }
        for pct in cumulative_percentages:
            metadata["statistics"]["classes"][class_name]["splits"]["subsets_fase_1"][f"pct_{pct}"] = 0

        test_size = int(math.ceil(actual_class_count * test_pct))
        test_images = selected_images[:test_size]
        train_pool_images = selected_images[test_size:]

        half_pool = len(train_pool_images) // 2
        fase_1_images = train_pool_images[:half_pool]
        fase_2_images = train_pool_images[half_pool:]

        metadata["statistics"]["classes"][class_name]["splits"]["global_test"] = len(test_images)
        metadata["statistics"]["classes"][class_name]["splits"]["train_fase_1"] = len(fase_1_images)
        metadata["statistics"]["classes"][class_name]["splits"]["train_fase_2"] = len(fase_2_images)
        metadata["statistics"]["classes"][class_name]["splits"]["fase_1_mais_fase_2"] = len(fase_1_images) + len(fase_2_images)

        metadata["statistics"]["splits"]["global_test"] += len(test_images)
        metadata["statistics"]["splits"]["train_fase_1"] += len(fase_1_images)
        metadata["statistics"]["splits"]["train_fase_2"] += len(fase_2_images)
        metadata["statistics"]["splits"]["fase_1_mais_fase_2"] += len(fase_1_images) + len(fase_2_images)
        metadata["statistics"]["total_images_processed"] += actual_class_count

        def copy_image(src_path, dest_dir, rel_copied_paths):
            dest_file_path = dest_dir / class_name / src_path.name
            rel_dest_path = dest_file_path.relative_to(output_path).as_posix()
            rel_copied_paths.append(rel_dest_path)

            (dest_dir / class_name).mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_path, dest_file_path)

        for img in selected_images:
            metadata["file_lineage"][img.name] = {
                "original_path": img.relative_to(input_path.parent).as_posix(),
                "class": class_name,
                "destinations": []
            }

        for img in test_images:
            copy_image(img, global_test_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "global_test"

        for img in fase_1_images:
            copy_image(img, train_fase_1_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "train_fase_1"
            copy_image(img, fase_1_plus_2_dir, metadata["file_lineage"][img.name]["destinations"])

        for img in fase_2_images:
            copy_image(img, train_fase_2_dir, metadata["file_lineage"][img.name]["destinations"])
            metadata["file_lineage"][img.name]["split"] = "train_fase_2"
            copy_image(img, fase_1_plus_2_dir, metadata["file_lineage"][img.name]["destinations"])

        for pct in cumulative_percentages:
            subset_size = int(round(len(fase_1_images) * (pct / 100.0)))
            subset_images = fase_1_images[:subset_size]

            metadata["statistics"]["classes"][class_name]["splits"]["subsets_fase_1"][f"pct_{pct}"] = len(subset_images)
            metadata["statistics"]["splits"]["subsets_fase_1"][f"pct_{pct}"] += len(subset_images)

            subset_pct_dir = subsets_dir / f"pct_{pct}"
            # The directory subset_pct_dir is now guaranteed to exist from the earlier block
            for img in subset_images:
                copy_image(img, subset_pct_dir, metadata["file_lineage"][img.name]["destinations"])

    metadata_file_path = output_path / "dataset_metadata.json"
    output_path.mkdir(parents=True, exist_ok=True)
    with open(metadata_file_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)

    print(f"\n[+] Separação Concluída. Estatísticas gravadas em: {metadata_file_path}")
    return True

print("A função 'process_dataset' foi definida. Ela será executada posteriormente no notebook, na célula 'MVq9awxZIomm'.")

A função 'process_dataset' foi definida. Ela será executada posteriormente no notebook, na célula 'MVq9awxZIomm'.


## Passo 4: Definição da Arquitetura ResNet-18 Customizada
Instanciada do zero, com controle do multiplicador de largura $\alpha$ e inicialização Xavier.

In [51]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ScalableResNet18(nn.Module):
    def __init__(self, num_classes: int = 10, alpha: float = 1.0):
        super(ScalableResNet18, self).__init__()
        self.in_planes = int(64 * alpha)

        self.conv1 = nn.Conv2d(3, self.in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(self.in_planes)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(BasicBlock, int(64 * alpha), 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, int(128 * alpha), 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, int(256 * alpha), 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, int(512 * alpha), 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(int(512 * alpha) * BasicBlock.expansion, num_classes)

        self._init_weights()

    def _make_layer(self, block, planes: int, num_blocks: int, stride: int):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.maxpool(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

## Passo 5: Classes de Dataset do PyTorch e Schedulers
Configuração de carregadores de dados e funções matemáticas de otimização passo a passo.

In [52]:
class TomatoDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        self.classes = sorted([d.name for d in root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.PNG", "*.JPEG")
        for cls_name in self.classes:
            cls_dir = root_dir / cls_name
            class_images = []
            for ext in extensions:
                class_images.extend(list(cls_dir.glob(ext)))
            for img_path in sorted(list(set(class_images))):
                self.image_paths.append(img_path)
                self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

def get_lr(step: int, total_steps: int, lr_max: float, warmup_pct: float = 0.05) -> float:
    warmup_steps = int(total_steps * warmup_pct)
    if step <= warmup_steps:
        return lr_max * (step / max(1, warmup_steps))
    else:
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * lr_max * (1.0 + math.cos(math.pi * progress))

def calculate_slope(y: List[float]) -> float:
    n = len(y)
    if n <= 1:
        return 0.0
    x = np.arange(n)
    sum_x = np.sum(x)
    sum_y = np.sum(y)
    sum_xx = np.sum(x**2)
    sum_xy = np.sum(x * y)
    denominator = (n * sum_xx) - (sum_x**2)
    return float(((n * sum_xy) - (sum_x * sum_y)) / denominator) if denominator != 0 else 0.0

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def evaluate(model: nn.Module, dataloader: DataLoader, device: torch.device) -> Tuple[float, float, float]:
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    acc = accuracy_score(all_targets, all_preds)
    return 1.0 - acc, f1_score(all_targets, all_preds, average="macro"), precision_score(all_targets, all_preds, average="macro", zero_division=0)

## Passo 6: Loop de Treinamento e Otimização por Steps

In [61]:
def train_single_run(
    seed: int,
    train_dir: Path,
    test_dir: Path,
    alpha: float,
    total_steps: int,
    batch_size: int,
    lr_max: float,
    weight_decay: float,
    img_size: int,
    device: torch.device,
    output_csv: str
) -> Tuple[int, float, float, float, float, float]:

    set_seed(seed)

    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = TomatoDataset(train_dir, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)

    # Check if the training dataset or loader is effectively empty and skip if so
    if len(train_dataset) == 0 or len(train_loader) == 0:
        print(f"  [WARN] Skipping training for Capacity: {alpha:.3f}, Subset: {train_dir.name}, Seed: {seed} - Dataset ou DataLoader vazio. len(train_dataset)={len(train_dataset)}, len(train_loader)={len(train_loader)}.")
        # Initialize a dummy model to calculate total_params, even if not trained
        model_for_params = ScalableResNet18(num_classes=10, alpha=alpha)
        total_params = sum(p.numel() for p in model_for_params.parameters())
        # Return dummy values for empty dataset runs:
        # dataset_size, total_params, loss_slope_final, test_error, f1_score_macro, precision_macro
        return 0, total_params, 0.0, 1.0, 0.0, 0.0


    test_dataset = TomatoDataset(test_dir, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    dataset_size = len(train_dataset)

    model = ScalableResNet18(num_classes=10, alpha=alpha).to(device)
    total_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    train_iter = iter(train_loader)
    loss_history = []

    model.train()
    for step in range(1, total_steps + 1):
        try:
            inputs, targets = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            inputs, targets = next(train_iter)

        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

        current_lr = get_lr(step, total_steps, lr_max)
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    slope_window = int(total_steps * 0.1)
    final_losses = loss_history[-slope_window:]
    loss_slope = calculate_slope(final_losses)

    test_error, f1, precision = evaluate(model, test_loader, device)

    # Append to CSV safely
    csv_file = Path(output_csv)
    fieldnames = ["seed", "tamanho_dataset", "fracao_parametros_modelo", "total_parametros", "loss_slope_final", "test_error", "f1_score_macro", "precision_macro"]
    file_exists = csv_file.exists()
    with open(csv_file, mode="a" if file_exists else "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            "seed": seed,
            "tamanho_dataset": dataset_size,
            "fracao_parametros_modelo": alpha,
            "total_parametros": total_params,
            "loss_slope_final": f"{loss_slope:.6e}",
            "test_error": f"{test_error:.6f}",
            "f1_score_macro": f"{f1:.6f}",
            "precision_macro": f"{precision:.6f}"
        })

    return dataset_size, total_params, loss_slope, test_error, f1, precision

## Passo 7: Modelagem Matemática de Leis de Escala
Funções para ajustar a equação teórica de Rosenfeld e Random Forest nos dados do Grid Search.

In [62]:
def rosenfeld_law(X: Tuple[np.ndarray, np.ndarray], a: float, b: float, c_inf: float, alpha: float, beta: float) -> np.ndarray:
    M, N = X
    M_val = np.maximum(M, 1e-5)
    N_val = np.maximum(N, 1e-5)
    return a * (M_val ** -alpha) + b * (N_val ** -beta) + c_inf

def fit_rosenfeld(df: pd.DataFrame) -> Tuple[tuple, bool]:
    M = df["fracao_parametros_modelo"].values
    N = df["tamanho_dataset"].values
    y = df["test_error"].values

    unique_points = len(df.groupby(["fracao_parametros_modelo", "tamanho_dataset"]).size())
    if unique_points < 5:
        return (0.15, 0.25, 0.02, 0.35, 0.28), False

    bounds = ((1e-6, 1e-6, 0.0, 1e-4, 1e-4), (10.0, 10.0, 1.0, 3.0, 3.0))
    p0 = [0.1, 0.2, 0.01, 0.5, 0.3]
    try:
        popt, _ = curve_fit(rosenfeld_law, (M, N), y, p0=p0, bounds=bounds, maxfev=10000)
        return tuple(popt), True
    except:
        return (0.15, 0.25, 0.02, 0.35, 0.28), False

def plot_surfaces(df: pd.DataFrame, popt: tuple, rf_model: RandomForestRegressor, mean_loss_slope: float):
    M_grid = np.linspace(0.1, 1.0, 50)
    N_grid = np.linspace(df["tamanho_dataset"].min(), df["tamanho_dataset"].max() * 2, 50)
    MM, NN = np.meshgrid(M_grid, N_grid)

    Z_rosenfeld = rosenfeld_law((MM, NN), *popt)

    grid_flat_M = MM.flatten()
    grid_flat_N = NN.flatten()
    grid_flat_slope = np.full_like(grid_flat_M, mean_loss_slope)
    X_grid_rf = np.column_stack((grid_flat_N, grid_flat_M, grid_flat_slope))
    Z_rf = rf_model.predict(X_grid_rf).reshape(MM.shape)

    fig = plt.figure(figsize=(16, 7))

    ax1 = fig.add_subplot(121, projection='3d')
    surf1 = ax1.plot_surface(MM, NN, Z_rosenfeld, cmap='viridis', alpha=0.8, edgecolor='none')
    ax1.scatter(df["fracao_parametros_modelo"], df["tamanho_dataset"], df["test_error"], color='red', s=50, label='Treinos Reais')
    ax1.set_title("Superfície: Ajuste de Rosenfeld")
    ax1.set_xlabel("Capacidade (M)")
    ax1.set_ylabel("Dataset (N)")
    ax1.set_zlabel("Erro")
    ax1.legend()

    ax2 = fig.add_subplot(122, projection='3d')
    surf2 = ax2.plot_surface(MM, NN, Z_rf, cmap='magma', alpha=0.8, edgecolor='none')
    ax2.scatter(df["fracao_parametros_modelo"], df["tamanho_dataset"], df["test_error"], color='red', s=50, label='Treinos Reais')
    ax2.set_title("Superfície: Random Forest")
    ax2.set_xlabel("Capacidade (M)")
    ax2.set_ylabel("Dataset (N)")
    ax2.set_zlabel("Erro")
    ax2.legend()

    plt.tight_layout()
    plt.show()

## Passo 8: Orquestrador do Experimento Completo
Esta célula une todas as fases e executa as 140 rodadas em sequência, seguido da extrapolação e validação do teste real na Fase 1+2.

In [ ]:
# --- Parâmetros Globais ---
ORIGINAL_DATA_DIR = "/content/dataset_original" # Pasta onde foi extraído o dataset original
PROCESSED_DATA_DIR = "/content/dataset_processed"

GRID_STEPS = 1000        # Orçamento de steps para o Grid Search (padrão rápido)
TARGET_STEPS = 5000      # Steps para o modelo final na base Fase 1+2
IMG_SIZE = 128           # Resolução das imagens
MAX_IMAGES_PER_CLASS = 1000
SEEDS = [42, 43, 44, 45, 46]

training_csv = "training_results.csv"
validation_csv = "validation_results.csv"

if os.path.exists(training_csv):
    os.remove(training_csv)
if os.path.exists(validation_csv):
    os.remove(validation_csv)

print("┌────────────────────────────────────────────────────────────┐")
print("│  INICIANDO PIPELINE DE LEIS DE ESCALA (SCALING LAWS)      │")
print("└────────────────────────────────────────────────────────────┘")

# 1. Engenharia de Dados
print("\n[Fase 1] Particionando e Balanceando Dataset...")
success = process_dataset(ORIGINAL_DATA_DIR, PROCESSED_DATA_DIR, seed=42, max_images_per_class=MAX_IMAGES_PER_CLASS, test_pct=0.15)

if not success:
    print("[!] Pipeline encerrado por erro de dados. Faça upload das imagens originais antes de rodar.")
else:
    # 2. Grid Search (140 Treinamentos)
    print("\n[Fase 2] Iniciando Grid Search (4 capacidades x 7 subsets x 5 seeds)... ")
    capacities = [0.125, 0.25, 0.5, 1.0]
    subsets = [1, 2, 5, 10, 20, 50, 100]

    total_runs = len(capacities) * len(subsets)
    run_idx = 1
    start_time = time.time()

    for alpha in capacities:
        for pct in subsets:
            print(f"  -> Executando [{run_idx}/{total_runs}] | Capacidade: {alpha:.3f} | Subset: {pct}%")

            # Rodar o treino para as 5 seeds desta configuração e gravar no CSV
            for seed in SEEDS:
                train_single_run(
                    seed=seed,
                    train_dir=Path(PROCESSED_DATA_DIR) / "subsets_fase_1" / f"pct_{pct}",
                    test_dir=Path(PROCESSED_DATA_DIR) / "global_test",
                    alpha=alpha,
                    total_steps=GRID_STEPS,
                    batch_size=64,
                    lr_max=0.002,
                    weight_decay=1e-4,
                    img_size=IMG_SIZE,
                    device=device,
                    output_csv=training_csv
                )
            run_idx += 1

    print(f"\n[+] Grid Search concluído! Tempo total: {(time.time() - start_time)/60:.1f} minutos.")

    # 3. Modelagem e Projeção Cega
    print("\n[Fase 3] Ajustando Meta-Modelos de Scaling Laws...")
    df_results = pd.read_csv(training_csv)
    popt, fitted = fit_rosenfeld(df_results)

    # Ajustar Random Forest
    X_rf = df_results[["tamanho_dataset", "fracao_parametros_modelo", "loss_slope_final"]].values
    y_rf = df_results["test_error"].values
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_rf, y_rf)

    # Projeção
    TARGET_N = 8500  # Fase 1 + Fase 2
    TARGET_M = 1.0   # ResNet-18 padrão
    mean_slope = df_results["loss_slope_final"].mean()

    pred_rosenfeld = rosenfeld_law((TARGET_M, TARGET_N), *popt)
    pred_rf = rf_model.predict([[TARGET_N, TARGET_M, mean_slope]])[0]

    print("\n" + "="*60)
    print(f"   PREVISÃO CEGA: Taxa de Erro Estimada para N={TARGET_N}, M={TARGET_M}")
    print("="*60)
    print(f"   Erro Estimado (Lei de Rosenfeld): {pred_rosenfeld*100:.4f}%")
    print(f"   Erro Estimado (Random Forest):    {pred_rf*100:.4f}%")
    print("="*60)

    # 4. Treinamento na base consolidada (Teste Empírico)
    print("\n[Fase 4] Rodando Treinamento Real na base consolidada Fase 1+2 (8.500 imagens)... ")
    for seed in SEEDS:
        train_single_run(
            seed=seed,
            train_dir=Path(PROCESSED_DATA_DIR) / "fase_1_mais_fase_2",
            test_dir=Path(PROCESSED_DATA_DIR) / "global_test",
            alpha=TARGET_M,
            total_steps=TARGET_STEPS,
            batch_size=64,
            lr_max=0.002,
            weight_decay=1e-4,
            img_size=IMG_SIZE,
            device=device,
            output_csv=validation_csv
        )

    # 5. Validação Empírica
    print("\n[Fase 5] Validando Precisão das Projeções contra a Realidade...")
    df_val = pd.read_csv(validation_csv)
    actual_error = df_val["test_error"].mean()

    print("\n" + "="*60)
    print("                      RESULTADOS DE VALIDAÇÃO                     ")
    print("="*60)
    print(f"   Erro Real Médio Observado:        {actual_error*100:.4f}%")
    print(f"   Margem de Erro (Rosenfeld):       {abs(pred_rosenfeld - actual_error)*100:.4f}%")
    print(f"   Margem de Erro (Random Forest):    {abs(pred_rf - actual_error)*100:.4f}%")
    print("="*60)

    # Renderizar Superfícies 3D
    plot_surfaces(df_results, popt, rf_model, mean_slope)

┌────────────────────────────────────────────────────────────┐
│  INICIANDO PIPELINE DE LEIS DE ESCALA (SCALING LAWS)      │
└────────────────────────────────────────────────────────────┘

[Fase 1] Particionando e Balanceando Dataset...

[+] Separação Concluída. Estatísticas gravadas em: /content/dataset_processed/dataset_metadata.json

[Fase 2] Iniciando Grid Search (4 capacidades x 7 subsets x 5 seeds)... 
  -> Executando [1/28] | Capacidade: 0.125 | Subset: 1%
  [WARN] Skipping training for Capacity: 0.125, Subset: pct_1, Seed: 42 - Dataset ou DataLoader vazio. len(train_dataset)=40, len(train_loader)=0.
  [WARN] Skipping training for Capacity: 0.125, Subset: pct_1, Seed: 43 - Dataset ou DataLoader vazio. len(train_dataset)=40, len(train_loader)=0.
  [WARN] Skipping training for Capacity: 0.125, Subset: pct_1, Seed: 44 - Dataset ou DataLoader vazio. len(train_dataset)=40, len(train_loader)=0.
  [WARN] Skipping training for Capacity: 0.125, Subset: pct_1, Seed: 45 - Dataset ou DataLo